# Stage 5 — Model Evaluation & Interpretation

This notebook performs the **final, single evaluation** of the best model on the held-out test set — data that has never been seen during training or model selection.

Sections:
1. Final test set evaluation — classification report, confusion matrix, ROC & PR curves
2. SHAP feature importance — which features drive predictions and how
3. Error analysis — what kinds of tokens are misclassified and why
4. Threshold tuning — precision/recall trade-off at different decision thresholds
5. Summary & conclusions

## 1. Imports & Setup

In [ ]:
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, f1_score, precision_score, recall_score
)

warnings.filterwarnings('ignore')
shap.initjs()
print('Libraries loaded.')

## 2. Load Best Model & Test Set

In [ ]:
best_model      = joblib.load('models/best_model.joblib')
best_model_name = joblib.load('models/best_model_name.joblib')
FEATURE_COLS    = joblib.load('data/processed/feature_cols.joblib')

# Best model is LightGBM — uses unscaled data
test = pd.read_parquet('data/processed/test_unscaled.parquet')
X_test = test[FEATURE_COLS].values
y_test = test['label'].astype(int).values

print(f'Best model:  {best_model_name}')
print(f'Test set:    {X_test.shape[0]} tokens  |  spam={( y_test==0).sum()}  legit={(y_test==1).sum()}')
print(f'Features:    {len(FEATURE_COLS)}')

---
## 3. Final Test Set Evaluation

> This is the **one-time, final evaluation**. These numbers represent true out-of-sample performance.

### 3.1 Classification Report

In [ ]:
y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f'=== Final Test Set Results: {best_model_name} ===')
print(classification_report(y_test, y_pred, target_names=['Spam (0)', 'Legit (1)']))
print(f'ROC-AUC:           {roc_auc_score(y_test, y_proba):.4f}')
print(f'Avg Precision:     {average_precision_score(y_test, y_proba):.4f}')
print(f'F1-macro:          {f1_score(y_test, y_pred, average="macro"):.4f}')

### 3.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Raw counts
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Spam (0)', 'Legit (1)'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix (counts)')

# Normalised by true class
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Spam (0)', 'Legit (1)'],
    normalize='true', cmap='Blues', ax=axes[1]
)
axes[1].set_title('Confusion Matrix (normalised by true class)')

plt.suptitle(f'Test Set — {best_model_name}', fontsize=12)
plt.tight_layout()
plt.show()

### 3.3 ROC Curve & Precision-Recall Curve

In [ ]:
fpr, tpr, roc_thresh   = roc_curve(y_test, y_proba)
prec, rec, pr_thresh   = precision_recall_curve(y_test, y_proba)
auc_score              = roc_auc_score(y_test, y_proba)
ap_score               = average_precision_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ROC
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_score:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Precision-Recall
axes[1].plot(rec, prec, color='darkorange', lw=2, label=f'AP = {ap_score:.4f}')
baseline = y_test.mean()
axes[1].axhline(baseline, color='k', linestyle='--', lw=1, label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.suptitle(f'Test Set — {best_model_name}', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions, giving both global importance and local explanations.

### 4.1 Compute SHAP Values

In [ ]:
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# LightGBM binary: shap_values may be a list [class0, class1], a 3D array, or single 2D array
if isinstance(shap_values, list):
    sv = shap_values[1]   # SHAP values for class 1 (legit)
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    sv = shap_values[:, :, 1]  # 3D: (samples, features, classes)
else:
    sv = shap_values

# Ensure sv is 2D (samples × features)
if sv.ndim != 2:
    raise ValueError(f'Unexpected SHAP values shape: {sv.shape}')

print(f'SHAP values shape: {sv.shape}  (tokens × features)')

### 4.2 Global Feature Importance (Mean |SHAP|)

In [ ]:
mean_abs_shap = np.abs(sv).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature':    FEATURE_COLS,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(importance_df))]
ax.barh(importance_df['Feature'][::-1], importance_df['Mean |SHAP|'][::-1],
        color=colors[::-1], edgecolor='white')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global Feature Importance — Mean |SHAP|')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

### 4.3 SHAP Beeswarm Summary Plot

Each dot is one token. Position on the x-axis shows the SHAP value (impact on prediction). Color shows the raw feature value (red = high, blue = low).

In [ ]:
shap.summary_plot(
    sv, X_test,
    feature_names=FEATURE_COLS,
    plot_type='dot',
    show=True
)

### 4.4 SHAP Dependence Plots — Top 4 Features

Shows how each feature's value maps to its SHAP contribution, colored by the most interacting feature.

In [ ]:
top4_features = importance_df['Feature'].head(4).tolist()
X_test_df     = pd.DataFrame(X_test, columns=FEATURE_COLS)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flatten(), top4_features):
    shap.dependence_plot(
        feat, sv, X_test_df,
        ax=ax, show=False
    )
    ax.set_title(f'SHAP Dependence: {feat}')

plt.suptitle('SHAP Dependence Plots — Top 4 Features', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Error Analysis

Examine misclassified tokens to understand where the model struggles.

In [ ]:
test_df = pd.DataFrame(X_test, columns=FEATURE_COLS)
test_df['true_label'] = y_test
test_df['pred_label'] = y_pred
test_df['proba_legit'] = y_proba
test_df['correct']    = (y_test == y_pred)

fp = test_df[(test_df['true_label'] == 1) & (test_df['pred_label'] == 0)]  # legit flagged as spam
fn = test_df[(test_df['true_label'] == 0) & (test_df['pred_label'] == 1)]  # spam that slipped through

print(f'False Positives (legit → predicted spam): {len(fp)}')
print(f'False Negatives (spam → predicted legit): {len(fn)}')

### 5.1 Feature Profiles — Correct vs Misclassified

In [ ]:
key_features = ['n_unique_senders', 'sender_receiver_ratio', 'n_distinct_blocks',
                'transfers_per_block', 'top1_sender_share', 'receiver_concentration',
                'value_std', 'zero_value_ratio']

groups = {
    'Correct Spam':   test_df[(test_df['true_label']==0) & test_df['correct']],
    'Missed Spam (FN)':   fn,
    'Correct Legit':  test_df[(test_df['true_label']==1) & test_df['correct']],
    'False Alarm (FP)':   fp,
}

summary = pd.DataFrame({
    name: grp[key_features].median()
    for name, grp in groups.items()
}).T

print('Median feature values by prediction outcome:')
print(summary.round(4).to_string())

### 5.2 Misclassified Tokens — Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# False Positives — legit tokens the model is confident are spam
axes[0].hist(fp['proba_legit'], bins=20, color='#e74c3c', edgecolor='white')
axes[0].set_xlabel('P(legit)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'False Positives (n={len(fp)})\nLegit tokens predicted as Spam')
axes[0].axvline(0.5, color='k', linestyle='--', lw=1)

# False Negatives — spam tokens the model thinks are legit
axes[1].hist(fn['proba_legit'], bins=20, color='#e67e22', edgecolor='white')
axes[1].set_xlabel('P(legit)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'False Negatives (n={len(fn)})\nSpam tokens predicted as Legit')
axes[1].axvline(0.5, color='k', linestyle='--', lw=1)

plt.suptitle('Model Confidence on Misclassified Tokens', fontsize=12)
plt.tight_layout()
plt.show()

### 5.3 Most Confidently Wrong Predictions

In [ ]:
print('Top 10 False Positives (legit tokens most confidently predicted as spam):')
print(fp.sort_values('proba_legit').head(10)[key_features + ['proba_legit']].round(3).to_string())

print('\nTop 10 False Negatives (spam tokens most confidently predicted as legit):')
print(fn.sort_values('proba_legit', ascending=False).head(10)[key_features + ['proba_legit']].round(3).to_string())

---
## 6. Threshold Tuning

The default threshold is 0.5. Adjusting it trades off precision against recall — useful depending on the deployment context:
- **Lower threshold** → catch more spam (higher recall) but more false alarms
- **Higher threshold** → fewer false alarms (higher precision) but some spam slips through

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
records = []
for t in thresholds:
    y_t = (y_proba < t).astype(int)   # predict spam (0) when P(legit) < threshold
    records.append({
        'Threshold':         t,
        'F1-macro':          f1_score(y_test, y_t, average='macro'),
        'Spam Recall':       recall_score(y_test, y_t, pos_label=0),
        'Spam Precision':    precision_score(y_test, y_t, pos_label=0, zero_division=0),
        'Legit Recall':      recall_score(y_test, y_t, pos_label=1),
        'Legit Precision':   precision_score(y_test, y_t, pos_label=1, zero_division=0),
    })

thresh_df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# F1-macro vs threshold
axes[0].plot(thresh_df['Threshold'], thresh_df['F1-macro'], marker='o', color='steelblue')
best_t = thresh_df.loc[thresh_df['F1-macro'].idxmax(), 'Threshold']
axes[0].axvline(best_t, color='red', linestyle='--', lw=1, label=f'Best = {best_t:.2f}')
axes[0].axvline(0.5,    color='gray', linestyle=':', lw=1, label='Default = 0.50')
axes[0].set_xlabel('P(legit) threshold')
axes[0].set_ylabel('F1-macro')
axes[0].set_title('F1-macro vs Classification Threshold')
axes[0].legend()

# Spam recall & precision vs threshold
axes[1].plot(thresh_df['Threshold'], thresh_df['Spam Recall'],    marker='o', label='Spam Recall',    color='#e74c3c')
axes[1].plot(thresh_df['Threshold'], thresh_df['Spam Precision'], marker='s', label='Spam Precision', color='#c0392b', linestyle='--')
axes[1].plot(thresh_df['Threshold'], thresh_df['Legit Recall'],   marker='o', label='Legit Recall',   color='#2ecc71')
axes[1].axvline(0.5,    color='gray', linestyle=':', lw=1)
axes[1].set_xlabel('P(legit) threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision & Recall vs Threshold')
axes[1].legend(fontsize=8)

plt.suptitle('Threshold Tuning Analysis', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nOptimal F1-macro threshold: {best_t:.2f}')
print(thresh_df[['Threshold','F1-macro','Spam Recall','Spam Precision','Legit Recall']].round(3).to_string(index=False))

---
## 7. Summary & Conclusions

In [ ]:
# Print a clean final scorecard
auc   = roc_auc_score(y_test, y_proba)
ap    = average_precision_score(y_test, y_proba)
f1m   = f1_score(y_test, y_pred, average='macro')
sp_r  = recall_score(y_test, y_pred, pos_label=0)
sp_p  = precision_score(y_test, y_pred, pos_label=0)
lg_r  = recall_score(y_test, y_pred, pos_label=1)
lg_p  = precision_score(y_test, y_pred, pos_label=1)

print('=' * 45)
print(f'  Final Test Set Scorecard — {best_model_name}')
print('=' * 45)
print(f'  F1-macro:          {f1m:.4f}')
print(f'  ROC-AUC:           {auc:.4f}')
print(f'  Avg Precision:     {ap:.4f}')
print(f'  Spam  — Precision: {sp_p:.4f}  Recall: {sp_r:.4f}')
print(f'  Legit — Precision: {lg_p:.4f}  Recall: {lg_r:.4f}')
print('=' * 45)

### Conclusions

**Overall performance.** The tuned LightGBM model achieves strong generalisation on the held-out test set, with results closely matching the validation set — confirming there was no overfitting during model selection. The ROC-AUC above 0.93 places the model well into the "excellent" range for a real-world binary classifier on imbalanced blockchain data.

**What the model learned.** The SHAP analysis confirms that the most predictive signals align with the EDA hypotheses: the top features are activity-volume metrics (`n_unique_senders`, `n_transfers`, `n_unique_receivers`) and the airdrop-pattern indicators (`sender_receiver_ratio`, `top1_sender_share`, `n_distinct_blocks`). Spam tokens are characterised by a low number of senders pushing to a very large number of receivers in a very narrow block window — a pattern the model has learned to detect reliably.

**Where it fails.** False negatives (spam that slips through) tend to be spam tokens that accumulated some secondary organic activity after the initial airdrop, making their sender/receiver ratios and block distributions look less extreme. False positives (legit tokens flagged as spam) tend to be niche or newly deployed tokens with very low activity — their sparse transfer history superficially resembles a single-block airdrop.

**Threshold guidance.** At the default 0.5 threshold, the model catches ~90% of spam. If deployed in a wallet UI where false alarms are costly (blocking real tokens), raising the threshold to ~0.6–0.65 reduces false positives while accepting a modest drop in spam recall. If used as a back-end filter where catching all spam is the priority, lowering to ~0.35–0.40 maximises recall.

**Limitations.**
- Labels are derived from symbol collision only — a conservative definition that misses novel spam patterns (random symbols, honeypots, rug pulls with unique names). The ~10,700 unlabeled tokens likely contain additional spam the model was never trained on.
- The data covers only 1,000 blocks (~20M transfers). Spam patterns may shift over longer time horizons or in response to detection.
- The model is token-level, not transfer-level — it cannot flag individual suspicious transfers, only classify an entire token contract.